# 03. Feature Engineering

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 02 showed that no single column predicts failure well, but the **gaps between columns** do. A shipment with 2 hours of buffer to spare is in danger. A shipment with 30 hours to spare is not. The raw columns do not say that directly.

**Goal of this notebook:** build a small set of features that

1. carry more signal than the raw columns,
2. mean something to a logistics coordinator, so the app can explain a prediction in plain words,
3. move when the coordinator acts, so the app can show risk dropping after an intervention,
4. use only what is known while the shipment is still moving.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets us import our own code from src/

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## 1. Load the clean file and lock away the outcome columns

Four columns are only known after delivery. We separate them first so they cannot slip into a feature by accident.

In [2]:
clean = pd.read_csv(ROOT / "data" / "processed" / "aura_shipments.csv")

TARGET = "Shipment_Failure"
LEAKAGE_COLUMNS = ["Actual_Transit_Hours", "Late_Delivery", "Temperature_Excursion", "Handling_Damage"]
FAILURE_TYPES = ["Shipment_Failure", "Late_Delivery", "Temperature_Excursion", "Handling_Damage"]

inputs = clean.drop(columns=LEAKAGE_COLUMNS + [TARGET])   # what a coordinator can see
outcomes = clean[FAILURE_TYPES]                           # used only to score features
y = clean[TARGET]

print(f"Shipments: {len(clean):,}   Failure rate: {y.mean():.1%}")
print(f"Input columns ({inputs.shape[1]}):", list(inputs.columns))

Shipments: 5,000   Failure rate: 15.7%
Input columns (17): ['Shipment_ID', 'Place_of_Origin', 'Place_of_Destination', 'Distance_km', 'Transport_Mode', 'Handover_Count', 'Lead_Time_Days', 'Planned_Transit_Hours', 'Transit_Duration_Hours', 'Stock_Volume', 'Geopolitical_Risk_Score', 'Fuel_Price_Index', 'Carrier_Reliability_Score', 'Buffer_of_Package', 'Storage_standard', 'Cargo_Value_USD', 'Hospital_Urgency_Tier']


## 2. The three questions a coordinator asks

Every feature below answers one of three questions about a live shipment.

| Question | Columns involved |
|---|---|
| **Will the cold last?** | `Buffer_of_Package` against `Transit_Duration_Hours` (the current ETA) |
| **Will it arrive in time?** | `Lead_Time_Days` against `Transit_Duration_Hours` |
| **Is it already slipping?** | `Transit_Duration_Hours` against `Planned_Transit_Hours` |

A fourth feature combines the first two: **how long until the first thing goes wrong?**

## 3. Build the features

In [3]:
def add_features(df):
    """Add the engineered columns. Uses only inputs known during transit."""
    out = df.copy()
    eta = out["Transit_Duration_Hours"]            # current expected door-to-door time
    window = out["Lead_Time_Days"] * 24            # delivery window in hours
    plan = out["Planned_Transit_Hours"]
    buffer = out["Buffer_of_Package"]

    # Will the cold last?
    out["buffer_margin_hours"] = buffer - eta      # hours of cold left on arrival
    out["buffer_cover_ratio"] = buffer / eta       # 1.0 means the buffer just covers the trip

    # Will it arrive in time?
    out["schedule_slack_hours"] = window - eta     # hours it can still lose before it is late
    out["schedule_use_ratio"] = eta / window       # 1.0 means the whole window is used up

    # Is it already slipping?
    out["known_delay_hours"] = eta - plan          # delay already reported
    out["known_delay_ratio"] = eta / plan - 1      # the same delay as a share of the plan

    # Which limit is hit first? The smaller of the two margins above.
    out["tightest_margin_hours"] = np.minimum(out["buffer_margin_hours"], out["schedule_slack_hours"])

    # Plain yes/no warnings
    out["buffer_short_flag"] = (out["buffer_margin_hours"] <= 0).astype(int)
    out["eta_past_window_flag"] = (out["schedule_slack_hours"] <= 0).astype(int)
    return out


ENGINEERED = [
    "buffer_margin_hours", "buffer_cover_ratio",
    "schedule_slack_hours", "schedule_use_ratio",
    "known_delay_hours", "known_delay_ratio",
    "tightest_margin_hours",
    "buffer_short_flag", "eta_past_window_flag",
]

featured = add_features(inputs)
print("All engineered values are finite numbers:", bool(np.isfinite(featured[ENGINEERED]).all().all()))
featured[["Shipment_ID"] + ENGINEERED].head()

All engineered values are finite numbers: True


,Shipment_ID,buffer_margin_hours,buffer_cover_ratio,schedule_slack_hours,schedule_use_ratio,known_delay_hours,known_delay_ratio,tightest_margin_hours,buffer_short_flag,eta_past_window_flag
0,SC-10000,28.3,1.714646,35.28,0.528846,1.6,0.042105,28.30,0,0
1,SC-10001,24.9,2.163551,9.80,0.685897,1.8,0.091837,9.80,0,0
2,SC-10002,26.5,1.784024,21.88,0.607040,4.5,0.153584,21.88,0,0
3,SC-10003,14.4,1.555985,8.90,0.744253,0.2,0.007782,8.90,0,0
4,SC-10004,10.6,1.841270,7.56,0.625000,0.9,0.076923,7.56,0,0


### What each feature means

| Feature | Plain meaning | Formula | Which intervention moves it |
|---|---|---|---|
| `buffer_margin_hours` | Hours of cold left when it arrives | buffer minus ETA | Replenish dry ice (adds buffer), prioritise transfer (cuts ETA) |
| `buffer_cover_ratio` | How many times over the buffer covers the trip | buffer / ETA | Same as above |
| `schedule_slack_hours` | Hours it can still lose before it is late | window minus ETA | Prioritise transfer, reroute |
| `schedule_use_ratio` | Share of the delivery window already committed | ETA / window | Prioritise transfer, reroute |
| `known_delay_hours` | Delay already reported | ETA minus plan | Prioritise transfer |
| `known_delay_ratio` | Delay as a share of the plan | ETA / plan minus 1 | Prioritise transfer |
| `tightest_margin_hours` | Hours to spare before the first limit is hit, cold or clock | smaller of buffer margin and schedule slack | Whichever intervention fixes the nearer limit |
| `buffer_short_flag` | 1 if the buffer runs out by the ETA | margin at or below 0 | Replenish dry ice |
| `eta_past_window_flag` | 1 if the ETA uses the whole window | slack at or below 0 | Reroute or reallocate stock |

The last column is why these features matter for the demo. When the presenter clicks *Replenish Dry Ice*, the app adds hours to `Buffer_of_Package`, `buffer_margin_hours` goes up, and the predicted risk drops. A feature the coordinator cannot change would make the simulator pointless.

## 4. How much signal does each feature carry?

We score every feature on its own against each kind of failure, using a number called **AUC**.

- Pick one failed shipment and one safe shipment at random. AUC is the chance that the feature ranks them the right way round.
- **0.50** is a coin flip (useless). **1.00** is perfect.

Engineered features are marked with a star.

In [4]:
from sklearn.metrics import roc_auc_score

RAW_NUMERIC = [
    "Buffer_of_Package", "Carrier_Reliability_Score", "Lead_Time_Days", "Geopolitical_Risk_Score",
    "Transit_Duration_Hours", "Planned_Transit_Hours", "Distance_km", "Handover_Count",
    "Stock_Volume", "Fuel_Price_Index", "Cargo_Value_USD",
]


def strength(values, outcome):
    """AUC of one column against one outcome. Direction does not matter, so we fold it above 0.5."""
    auc = roc_auc_score(outcome, values)
    return max(auc, 1 - auc)


signal = pd.DataFrame({
    outcome: {col: strength(featured[col], outcomes[outcome]) for col in ENGINEERED + RAW_NUMERIC}
    for outcome in FAILURE_TYPES
}).sort_values("Shipment_Failure", ascending=True)
labels = [f"{name} *" if name in ENGINEERED else name for name in signal.index]

fig = go.Figure(go.Heatmap(
    z=signal.values, x=["Any failure", "Late delivery", "Temperature excursion", "Handling damage"], y=labels,
    zmin=0.5, zmax=0.9, colorscale=[[0, "#f0efec"], [1, "#1d5fb0"]],
    text=signal.values, texttemplate="%{text:.2f}", textfont=dict(size=11),
    xgap=2, ygap=2, colorbar=dict(title="AUC", thickness=12),
    hovertemplate="%{y}<br>%{x}<br>AUC: %{z:.3f}<extra></extra>",
))
fig.update_xaxes(side="top")
style(fig, "Signal in each feature on its own (AUC, 0.50 = coin flip)", height=720)
fig.update_layout(margin=dict(l=220, t=110))
fig.show()

best = pd.DataFrame({
    "best_raw_column": signal.loc[RAW_NUMERIC].idxmax(),
    "raw_auc": signal.loc[RAW_NUMERIC].max().round(3),
    "best_engineered_feature": signal.loc[ENGINEERED].idxmax(),
    "engineered_auc": signal.loc[ENGINEERED].max().round(3),
})
best["gain"] = (best["engineered_auc"] - best["raw_auc"]).round(3)
best

,best_raw_column,raw_auc,best_engineered_feature,engineered_auc,gain
Shipment_Failure,Buffer_of_Package,0.624,tightest_margin_hours,0.816,0.192
Late_Delivery,Carrier_Reliability_Score,0.658,schedule_use_ratio,0.817,0.159
Temperature_Excursion,Buffer_of_Package,0.758,buffer_margin_hours,0.845,0.087
Handling_Damage,Handover_Count,0.578,schedule_use_ratio,0.540,-0.038


**What this tells us:**

- For **temperature excursions**, `buffer_margin_hours` is much stronger than the raw buffer. Knowing how big the buffer is helps. Knowing how much of it is left over after the trip helps far more.
- For **late delivery**, the schedule features are far stronger than any raw column.
- For **any failure**, `tightest_margin_hours` is the strongest single feature, because it watches the cold and the clock at once.
- For **handling damage**, nothing scores well. It is rare and mostly random, with only a weak link to the number of handovers. We should be honest that the model will not predict it reliably.
- `Stock_Volume`, `Fuel_Price_Index` and `Cargo_Value_USD` sit at 0.50. They carry no signal about failure.

## 5. Do the features behave sensibly?

A good feature should show risk rising steadily as the situation gets worse. Here is the failure rate inside bands of the three main features.

In [5]:
panels = [
    ("buffer_margin_hours", [-999, 0, 6, 12, 24, 999], ["0 or less", "0 to 6", "6 to 12", "12 to 24", "Over 24"],
     "Buffer margin (hours)", "Temperature_Excursion", "Excursion rate"),
    ("schedule_slack_hours", [-999, 0, 3, 6, 12, 999], ["0 or less", "0 to 3", "3 to 6", "6 to 12", "Over 12"],
     "Schedule slack (hours)", "Late_Delivery", "Late rate"),
    ("known_delay_ratio", [-1, 0.05, 0.10, 0.20, 0.40, 99], ["Under 5%", "5 to 10%", "10 to 20%", "20 to 40%", "Over 40%"],
     "Known delay (share of plan)", "Shipment_Failure", "Failure rate"),
]

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{p[3]}<br>{p[5]}" for p in panels], horizontal_spacing=0.07)
band_tables = {}
for i, (col, edges, names, title, outcome, _) in enumerate(panels, start=1):
    bands = pd.cut(featured[col], bins=edges, labels=names)
    grouped = outcomes.groupby(bands, observed=False)[outcome].agg(["mean", "size"])
    band_tables[title] = grouped.rename(columns={"mean": "rate", "size": "shipments"})
    fig.add_trace(go.Bar(
        x=grouped.index.astype(str), y=grouped["mean"], marker_color=BLUE,
        text=[f"{m:.0%}" for m in grouped["mean"]], textposition="outside",
        textfont=dict(color=INK), cliponaxis=False, customdata=grouped["size"],
        hovertemplate="%{x}<br>Rate: %{y:.1%}<br>Shipments: %{customdata:,}<extra></extra>",
    ), row=1, col=i)
fig.update_annotations(font=dict(size=12, color=INK))
fig.update_yaxes(tickformat=".0%", range=[0, 1.15])
fig.update_xaxes(tickangle=-30)
style(fig, "Risk rises steadily as each feature gets worse", height=460)
fig.update_layout(showlegend=False, bargap=0.3, margin=dict(t=110, b=90))
fig.show()

pd.concat(band_tables).round(3)

rate  shipments
Buffer margin (hours)       0 or less  1.000        125
                            0 to 6     0.544        160
                            6 to 12    0.114        403
                            12 to 24   0.038       2194
                            Over 24    0.023       2118
Schedule slack (hours)      0 or less  1.000        104
                            0 to 3     0.398        467
                            3 to 6     0.106        812
                            6 to 12    0.048       1759
                            Over 12    0.035       1858
Known delay (share of plan) Under 5%   0.102       2194
                            5 to 10%   0.112       1158
                            10 to 20%  0.167       1040
                            20 to 40%  0.361        501
                            Over 40%   0.738        107

**What this tells us:** each feature moves in one direction, with no odd bumps. Less margin means more excursions. Less slack means more late deliveries. More delay already on the clock means more failures of every kind. This is the behaviour the app needs to give a coordinator a believable explanation.

## 6. Are any features saying the same thing twice?

Two features that are almost identical add no information and make explanations confusing. We list the pairs whose correlation is above 0.80.

In [6]:
numeric_candidates = ENGINEERED + [c for c in RAW_NUMERIC if c != "Cargo_Value_USD"]
corr = featured[numeric_candidates].corr()

pairs = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    .stack().dropna().rename("correlation").reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)
pairs = pairs[pairs["correlation"].abs() > 0.80].sort_values("correlation", key=abs, ascending=False)
pairs.round(2).reset_index(drop=True)

,feature_a,feature_b,correlation
0,Transit_Duration_Hours,Planned_Transit_Hours,0.96
1,Lead_Time_Days,Planned_Transit_Hours,0.92
2,known_delay_hours,known_delay_ratio,0.91
3,schedule_slack_hours,tightest_margin_hours,0.89
4,Lead_Time_Days,Transit_Duration_Hours,0.88
5,buffer_margin_hours,Buffer_of_Package,0.84
6,schedule_slack_hours,Lead_Time_Days,0.80


**What this tells us:**

- The overlaps are expected. Some pairs are built from each other (a margin and the raw column it starts from, the hours and ratio versions of the delay). The rest are time columns that all grow with the length of the trip (planned time, current ETA, delivery window).
- None of the pairs is a perfect copy, so each still adds something. We keep both the hours and the ratio versions. Hours are what a coordinator reads ("2.8 hours of buffer left"). Ratios let the model compare a 6-hour truck run fairly with a 40-hour long-haul flight. Tree-based models are not harmed by overlapping features.
- For a simple linear model, notebook 04 should check whether dropping one of each pair changes anything.

## 7. Text columns

Models need numbers, so text columns are turned into yes/no columns, one per category. This is called one-hot encoding.

- `Transport_Mode` and `Storage_standard` are encoded and given to the model.
- `Hospital_Urgency_Tier` is tested below. Urgency already shapes the delivery window, so it may add nothing new.
- `Shipment_ID`, origin and destination are labels, not causes. They are left out so the model cannot memorise individual routes.
- `Cargo_Value_USD` is left out of the model on purpose. It showed no link to failure, and it is used later to rank shipments and to show money saved.

In [7]:
CATEGORICAL = ["Transport_Mode", "Storage_standard"]
BASE_NUMERIC = [
    "Distance_km", "Planned_Transit_Hours", "Transit_Duration_Hours", "Lead_Time_Days",
    "Buffer_of_Package", "Handover_Count", "Carrier_Reliability_Score",
    "Geopolitical_Risk_Score", "Fuel_Price_Index", "Stock_Volume",
]

encoded = pd.get_dummies(featured[CATEGORICAL]).astype(int)
urgency = pd.get_dummies(featured[["Hospital_Urgency_Tier"]]).astype(int)
print("Encoded columns:", list(encoded.columns))
encoded.head(3)

Encoded columns: ['Transport_Mode_Air', 'Transport_Mode_Land', 'Transport_Mode_Multi-Modal Air-to-Land', 'Storage_standard_-20°C', 'Storage_standard_2°C to 8°C', 'Storage_standard_Deep Frozen / Dry Ice']


,Transport_Mode_Air,Transport_Mode_Land,Transport_Mode_Multi-Modal Air-to-Land,Storage_standard_-20°C,Storage_standard_2°C to 8°C,Storage_standard_Deep Frozen / Dry Ice
0,0,0,1,0,1,0
1,0,0,1,0,1,0
2,0,0,1,0,0,1


## 8. Do the features help a model?

Scoring features one at a time is not the whole story. Here we train two quick models on four different sets of columns and compare them.

**The four sets**
1. **Raw inputs only:** the clean columns as they are.
2. **Current pipeline:** what `src/features.py` produces today.
3. **Proposed:** raw inputs plus the nine features from this notebook.
4. **Proposed plus urgency tier.**

**The two models**
- **Logistic regression:** a simple model that adds up weighted inputs.
- **Gradient boosting:** a stronger model built from many small decision trees.

**The two scores** (both averaged over 5 rounds, each round tested on shipments the model did not train on)
- **ROC AUC:** 0.50 is a coin flip, 1.00 is perfect.
- **Average precision:** how clean the top of the risk list is. A random list would score about 0.16, the failure rate.

This is a quick comparison of feature sets, not model selection. Choosing and tuning the model is the job of notebook 04.

In [8]:
import warnings

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data_loader import load_shipments
from src.features import extract_features

# What the app's own pipeline builds today from the same file
current = extract_features(load_shipments(ROOT / "data" / "processed" / "aura_shipments.csv"))
current_numeric = current.select_dtypes("number").reset_index(drop=True)
current_engineered = [c for c in current_numeric.columns if c[0].islower()]
print("Engineered by src/features.py today:", current_engineered)

feature_sets = {
    "1. Raw inputs only": pd.concat([featured[BASE_NUMERIC], encoded], axis=1),
    "2. Current pipeline (src/features.py)": pd.concat([current_numeric, encoded], axis=1),
    "3. Proposed": pd.concat([featured[BASE_NUMERIC + ENGINEERED], encoded], axis=1),
    "4. Proposed plus urgency tier": pd.concat([featured[BASE_NUMERIC + ENGINEERED], encoded, urgency], axis=1),
}
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for set_name, X in feature_sets.items():
        for model_name, model in models.items():
            scores = cross_validate(model, X, y, cv=folds, scoring=["roc_auc", "average_precision"])
            rows.append({
                "feature_set": set_name, "model": model_name, "columns": X.shape[1],
                "roc_auc": scores["test_roc_auc"].mean(), "roc_auc_spread": scores["test_roc_auc"].std(),
                "avg_precision": scores["test_average_precision"].mean(),
                "avg_precision_spread": scores["test_average_precision"].std(),
            })
comparison = pd.DataFrame(rows)
comparison.round(3)

Engineered by src/features.py today: ['remaining_sla_hours', 'thermal_margin_to_deadline', 'elapsed_sla_ratio', 'buffer_to_deadline_ratio', 'multimodal_flag']


,feature_set,model,columns,roc_auc,roc_auc_spread,avg_precision,avg_precision_spread
0,1. Raw inputs only,Logistic regression,16,0.798,0.012,0.569,0.026
1,1. Raw inputs only,Gradient boosting,16,0.830,0.023,0.688,0.033
2,2. Current pipeline (src/features.py),Logistic regression,19,0.822,0.010,0.675,0.010
3,2. Current pipeline (src/features.py),Gradient boosting,19,0.832,0.016,0.705,0.018
4,3. Proposed,Logistic regression,25,0.841,0.014,0.708,0.015
5,3. Proposed,Gradient boosting,25,0.841,0.015,0.719,0.023
6,4. Proposed plus urgency tier,Logistic regression,28,0.840,0.014,0.707,0.016
7,4. Proposed plus urgency tier,Gradient boosting,28,0.841,0.015,0.719,0.023


In [9]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["ROC AUC", "Average precision"], horizontal_spacing=0.08,
                    shared_yaxes=True)
model_colors = {"Logistic regression": BLUE, "Gradient boosting": ORANGE}
for col, (metric, spread) in enumerate([("roc_auc", "roc_auc_spread"), ("avg_precision", "avg_precision_spread")], start=1):
    for model_name, color in model_colors.items():
        part = comparison[comparison["model"] == model_name]
        fig.add_trace(go.Scatter(
            x=part[metric], y=part["feature_set"], mode="markers", name=model_name,
            legendgroup=model_name, showlegend=col == 1,
            marker=dict(color=color, size=11, line=dict(color=SURFACE, width=2)),
            error_x=dict(type="data", array=part[spread], color=color, thickness=1.5, width=4),
            hovertemplate=model_name + "<br>%{y}<br>%{x:.3f}<extra></extra>",
        ), row=1, col=col)
fig.update_yaxes(autorange="reversed")
fig.update_annotations(font=dict(size=12, color=INK))
style(fig, "Feature sets compared (dot = average of 5 rounds, bar = spread)", height=400)
fig.update_layout(legend=dict(orientation="h", y=-0.2, x=0, font=dict(color=INK)), margin=dict(l=260, b=90))
fig.show()

**What this tells us, stated carefully:**

- **The simple model gains the most.** With raw inputs only, logistic regression cannot see the gaps between columns. With the proposed features it catches up with gradient boosting on ROC AUC (the two dots sit on top of each other in the chart). A simple model that performs as well as a complex one is easier to explain to a coordinator.
- **The strong model gains a little.** Gradient boosting can work out some of these gaps by itself, so its improvement is small and about the size of the round-to-round spread. We should not oversell it.
- **Against the current pipeline, the proposed set is ahead on both models.** The lead is clear for logistic regression and small for gradient boosting. The stronger reason to switch is meaning, covered in the next section.
- **Urgency tier adds nothing** once the schedule features are in. It stays out of the model and is used in the triage ranking instead.

## 9. Why the current pipeline's features need updating

`src/features.py` was written when two columns had a different meaning:

| Column | Meaning when `src/features.py` was written | Meaning in the clean file |
|---|---|---|
| `Transit_Duration_Hours` | Hours elapsed so far | Expected total door-to-door time (current ETA) |
| `Buffer_of_Package` | Hold time left right now | Hold time counted from dispatch |

With the new meanings, some of the current features no longer describe anything real:

- `thermal_margin_to_deadline` is buffer minus (window minus ETA). That subtracts a leftover schedule gap from a total hold time, which is not a quantity a coordinator would recognise.
- `buffer_to_deadline_ratio` has the same problem.
- `remaining_sla_hours` and `elapsed_sla_ratio` still work. They are the same numbers as `schedule_slack_hours` and `schedule_use_ratio` here, under names that no longer fit.

The current pipeline also ignores `Planned_Transit_Hours` and `Handover_Count`, so it cannot see how much delay has already been reported.

In [10]:
# Same numbers, different names: the two schedule features match the current pipeline's
same_slack = np.allclose(current_numeric["remaining_sla_hours"], featured["schedule_slack_hours"].clip(lower=0))
same_use = np.allclose(current_numeric["elapsed_sla_ratio"], featured["schedule_use_ratio"])
print(f"remaining_sla_hours equals schedule_slack_hours (floored at 0): {same_slack}")
print(f"elapsed_sla_ratio equals schedule_use_ratio                   : {same_use}")

excursion = outcomes["Temperature_Excursion"]
print()
print("Signal for temperature excursions (AUC):")
print(f"  current  thermal_margin_to_deadline : {strength(current_numeric['thermal_margin_to_deadline'], excursion):.3f}")
print(f"  proposed buffer_margin_hours        : {strength(featured['buffer_margin_hours'], excursion):.3f}")

remaining_sla_hours equals schedule_slack_hours (floored at 0): True
elapsed_sla_ratio equals schedule_use_ratio                   : True

Signal for temperature excursions (AUC):
  current  thermal_margin_to_deadline : 0.744
  proposed buffer_margin_hours        : 0.845


**Recommendation:** move `add_features` from this notebook into `src/features.py` and update the feature list in `src/schema.py`, so that training, the app and the explanations all use the same names. That change touches shared files, so the team should agree on it first.

## 10. Final check and feature list

In [11]:
MODEL_NUMERIC = BASE_NUMERIC + ENGINEERED
MODEL_CATEGORICAL = CATEGORICAL
MODEL_FEATURES = MODEL_NUMERIC + MODEL_CATEGORICAL

leaked = set(MODEL_FEATURES) & set(LEAKAGE_COLUMNS + [TARGET])
assert not leaked, f"Outcome columns found in the feature list: {leaked}"

model_table = featured[MODEL_FEATURES]
print(f"No outcome columns in the feature list : True")
print(f"Missing values in the feature table    : {int(model_table.isna().sum().sum())}")
print(f"Numeric features ({len(MODEL_NUMERIC)})  : {MODEL_NUMERIC}")
print(f"Text features ({len(MODEL_CATEGORICAL)})      : {MODEL_CATEGORICAL}")
print(f"Left out on purpose      : Shipment_ID, Place_of_Origin, Place_of_Destination, Cargo_Value_USD, Hospital_Urgency_Tier")
model_table.describe().T.round(2)

No outcome columns in the feature list : True
Missing values in the feature table    : 0
Numeric features (19)  : ['Distance_km', 'Planned_Transit_Hours', 'Transit_Duration_Hours', 'Lead_Time_Days', 'Buffer_of_Package', 'Handover_Count', 'Carrier_Reliability_Score', 'Geopolitical_Risk_Score', 'Fuel_Price_Index', 'Stock_Volume', 'buffer_margin_hours', 'buffer_cover_ratio', 'schedule_slack_hours', 'schedule_use_ratio', 'known_delay_hours', 'known_delay_ratio', 'tightest_margin_hours', 'buffer_short_flag', 'eta_past_window_flag']
Text features (2)      : ['Transport_Mode', 'Storage_standard']
Left out on purpose      : Shipment_ID, Place_of_Origin, Place_of_Destination, Cargo_Value_USD, Hospital_Urgency_Tier


,count,mean,std,min,25%,50%,75%,max
Distance_km,5000.0,7429.28,3881.06,101.00,5173.75,9190.00,9998.00,14929.00
Planned_Transit_Hours,5000.0,23.62,8.26,3.70,18.70,23.70,28.40,61.60
Transit_Duration_Hours,5000.0,25.82,9.26,3.70,20.30,25.65,31.10,82.30
Lead_Time_Days,5000.0,1.53,0.59,0.18,1.17,1.50,1.87,4.95
Buffer_of_Package,5000.0,48.62,16.30,18.40,41.30,45.40,62.70,119.80
Handover_Count,5000.0,3.18,1.06,1.00,2.00,3.00,4.00,5.00
Carrier_Reliability_Score,5000.0,0.75,0.14,0.50,0.63,0.76,0.88,1.00
Geopolitical_Risk_Score,5000.0,5.08,2.88,0.00,2.60,5.10,7.50,10.00
Fuel_Price_Index,5000.0,2.85,0.96,1.20,2.02,2.84,3.71,4.50
Stock_Volume,5000.0,2601.77,2515.35,250.00,621.00,1535.50,3948.00,9988.00


## What we found and what happens next

### Findings

| # | Finding | What we do about it |
|---|---|---|
| 1 | Gaps between columns carry far more signal than the columns themselves | Use the nine engineered features |
| 2 | `buffer_margin_hours` is the best single signal for temperature excursions | Show it on screen as "hours of thermal runway" |
| 3 | The schedule features are the best signals for late delivery | Use them to explain delay risk |
| 4 | `tightest_margin_hours` is the best single signal for failure of any kind | A natural number to sort or highlight in the triage queue |
| 5 | Handling damage cannot be predicted well from these inputs | Say so, and do not promise it in the pitch |
| 6 | The features let a simple model match a complex one | Notebook 04 should compare both fairly |
| 7 | The gain for the strong model is small | Report it honestly |
| 8 | Urgency tier and cargo value do not help prediction | Use them for ranking and money saved |
| 9 | `src/features.py` was written for older column meanings | Agree as a team, then update it |

### Next

Notebook 04 trains and compares models on `MODEL_FEATURES`, picks a decision threshold that keeps false alarms low, and reports precision and recall.

### Limits we state openly

- Outcomes are simulated from rules that use buffer, delivery window, delay, carrier reliability and handovers. Features built from those same columns are bound to score well. This shows the features recover the built-in pattern. It does not show they would work on a real carrier's records.
- The feature comparison uses default model settings and one random split into 5 rounds.